In [ ]:
"2025-08-28"
"This program tests the CORE gripper arm to move plates."
"It has been used to successfully move COSTAR plate from 13[0]-->13[1]. Gripper strength = 30"
"And BioER DW plate 7[0]-->7[1] in empty and full(175g) configurations. Gripper strength = 30, 65, resp."


'And BioER DW plate 7[0]-->7[1] in empty and full(175g) configurations. Gripper strength = 30, 65, resp.'

In [ ]:
###############################################################################
# 0) SETUP (unchanged)
###############################################################################
%load_ext autoreload
%autoreload 2
import asyncio

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import (
    STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
)
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
# from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
# from pylabrobot.resources.corning.plates import Cor_96_wellplate_360ul_Fb
from pylabrobot.resources.diy.grindbio import Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint

# from pylabrobot.resources.celltreat.plates import CellTreat_96_wellplate_350ul_Ub
from pylabrobot.resources import (
    HTF,             # 300 µL filtered tips
    TIP_50ul_w_filter,        # 50 µL filtered tips
    LTF
)


In [3]:

###############################################################################
# 1) BUILD LH + DECK
###############################################################################
backend = STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())
await lh.setup(skip_autoload=True)        # faster while iterating

In [ ]:
from typing import Optional

from pylabrobot.resources.height_volume_functions import (
  compute_height_from_volume_rectangle,
  compute_volume_from_height_rectangle,
)
from pylabrobot.resources.plate import Lid, Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200ul
##############################################################################
#2) CARRIERS & LABWARE
##############################################################################

# # tips
tip_car        = TIP_CAR_480_A00("tip_car")
tip_car[0]     = HTF(name="htf_tips")       # 300 µL
tip_car[1]     = TIP_50ul_w_filter(name="htf_50ul")  # 50 µL
tip_car[2]     = LTF(name="htf_10ul")  # 50 µL
lh.deck.assign_child_resource(tip_car, rails=25)
htf_tips  = lh.deck.get_resource("htf_tips")   # 1000 µL filtered rack
tips_50ul = lh.deck.get_resource("htf_50ul")   # 50 µL filtered rack
tips_10ul = lh.deck.get_resource("htf_10ul")   # 50 µL filtered rack

# --- SOURCE CARRIER (MFX) --- orange G
src_mod  = Hamilton_MFX_plateholder_DWP_metal_tapped("src_mod")
src_car  = MFX_CAR_L5_base("src_car", modules={0: src_mod})
lh.deck.assign_child_resource(src_car, rails=19)

# src_plate = AGenBio_1_troughplate_100000uL_Fl("source_plate")
# src_mod.assign_child_resource(src_plate)

# --- DESTINATION + WATER (MFX) ---
dst_mod   = Hamilton_MFX_plateholder_DWP_metal_tapped("dst_mod")
water_mod = Hamilton_MFX_plateholder_DWP_metal_tapped("water_mod")

dst_car = MFX_CAR_L5_base("dst_car", modules={0: dst_mod, 1: water_mod})
lh.deck.assign_child_resource(dst_car, rails=13)

# dst_plate   = CellTreat_96_wellplate_350ul_Ub("dest_plate")
dst_plate   = Cor_96_wellplate_360ul_Fb("dest_plate")
# water_plate = AGenBio_1_troughplate_100000uL_Fl("water_plate")

dst_mod.assign_child_resource(dst_plate)

# CARRIER-->MODULE-->PLATES
carrier7 = MFX_CAR_L5_base(
    "carrier7",
    modules={
        0: Hamilton_MFX_plateholder_DWP_metal_tapped("c7_m0"), #carrier7 module[0]
        1: Hamilton_MFX_plateholder_DWP_metal_tapped("c7_m1"),
    }
)
lh.deck.assign_child_resource(carrier7, rails=7)
carrier7[0].assign_child_resource(BioER_96_wellplate_Vb_2200ul("c7_plate"))


In [5]:
# --- move the plate from site 0 → site 1 on the *same* carrier -------------
src  = carrier7[0].children[0]          # the plate you want to move
dest = carrier7[1]       # an empty site on that carrier
# src  = carrier7[1].children[0]          # the plate you want to move
# dest = carrier7[0]       # an empty site on that carrier

await lh.move_plate(                       # high-level helper
    src,
    dest,
    use_arm="core",                        # tell PLR to use the CO-RE gripper
    channel_1=2, channel_2=3,              # zero-based channels that hold the fingers
    core_grip_strength=30,                 # 0 = loose … 99 = very tight
    return_core_gripper=True               # park the fingers afterwards
)

STARFirmwareError: {'Pipetting channel 2': HamiltonNoTipError('No tip picked up, possibly because no was present at specified position'), 'Pipetting channel 3': HamiltonNoTipError('No tip picked up, possibly because no was present at specified position')}, C0ZTid0007er99/00 P208/75 P308/75

In [ ]:
# src  = dst_plate          # the plate you want to move
# dest = water_mod          # an empty site on that carrier

# await lh.move_plate(                       # high-level helper
#     src,
#     dest,
#     use_arm="core",                        # tell PLR to use the CO-RE gripper
#     channel_1=2, channel_2=3,              # zero-based channels that hold the fingers, 
#     core_grip_strength=30,                 # 0 = loose … 99 = very tight
#     return_core_gripper=True               # park the fingers afterwards
# )

In [ ]:
# ###############################################################################
# # 3) HELPERS
# ###############################################################################
# CHANNELS_8 = list(range(8))
# ROWS = "ABCDEFGH"

# def col_wells(plate, col: int):
#     # """Return [A{col}..H{col}] for an ANSI plate."""
#     # print ("The plate wells are:")
#     # print ([plate[f"{r}{col}"] for r in ROWS])
#     # containers = plate[f"{r}{col}"] for r in ROWS
#     # return containers
#     return [plate[f"{r}{col}"][0] for r in ROWS]
    

# ###############################################################################
# # 4) PROTOCOL
# ###############################################################################
# async def fill_plate_with_water():
#     # --- STEP 1: HTF (1000 µL) tips on all channels; prewet and aspirate 1000 µL water ---
#     # Pick up a full column of HTF tips (A1..H1)
#     # don't forget to change "A1:H1" location below.
#     await lh.pick_up_tips(htf_tips["A2:H2"], use_channels=CHANNELS_8)  # HTF = 1000 µL filtered
 
#     precise_trough = dict(
#     lld_mode=[STARBackend.LLDMode.GAMMA]*8,
#     gamma_lld_sensitivity=[2]*8,          # tweak if needed
#     immersion_depth=[1]*8,
#     immersion_depth_direction=[0]*8,
#     surface_following_distance=[1]*8,
#     transport_air_volume=[0]*8,
#     settling_time=[1]*8
#     # dispense_position_above_z_touch_off=[2]*8,
#     # side_touch_off_distance=0,            # avoid edge “scrub” on a single big well
# )

#     # await lh.aspirate(water_plate["A1"]*8, vols=[1000]*8, use_channels=CHANNELS_8, lld_mode=[STARBackend.LLDMode.GAMMA]*8)
#     # await lh.dispense(water_plate["A1"]*8, vols=[1000]*8, use_channels=CHANNELS_8, lld_mode=[STARBackend.LLDMode.GAMMA]*8)
  
#     # We'll dispense water in 3 blocks of 4 columns (1–4, 5–8, 9–12), re-aspirating each block
#     for block_start in (1, 5, 9):
#         block_cols = range(block_start, block_start + 4)

#         # Fresh 1000 µL per channel for each 4-column block
#         await lh.aspirate(
#             water_plate["A1"]*8, #[0]#when you want to spread multiple channels into a single container (like one big trough),
#             vols=[950]*8, 
#             use_channels=CHANNELS_8,
#             **precise_trough)
        
#         await lh.dispense(water_plate["A1"]*8, vols=[100]*8, use_channels=CHANNELS_8, **precise_trough)
        
#         # --- STEP 2: Dispense 200 µL to each column in this block ---
#         for c in block_cols:
#             dests = col_wells(dst_plate, c)
#             # print ("The dests are:", dests)
#             await lh.dispense(
#                 dests,
#                 vols=[200]*8,
#                 use_channels=CHANNELS_8,
#                 liquid_height=[5]*8,
#                 transport_air_volume=[0]*8,
#                 settling_time=[1]*8, 
#                 flow_rates=[60]*8
#                 )

#         # Clear ~200 µL remainder per channel back to water to avoid drip carryover
#         # I know that 100 + 200*4 + 20 <> 950, but robot complains with other values.
#         # 20ul in final was empirically determined. Prob due to air gaps, etc.
    
#         # await lh.dispense(water_plate["A1"]*8, vols=[20]*8, use_channels=CHANNELS_8, **precise_trough, blow_out=[1]*8)
#         await lh.dispense(water_plate["A1"]*8, vols=[0]*8, use_channels=CHANNELS_8, **precise_trough, blow_out=[1]*8)

#     # --- STEP 3: Return HTF tips (for reuse later if desired) ---
#     # await lh.return_tips()  # returns to original rack positions
#     # await lh.drop_tips(htf_tips["A2:H2"], use_channels=CHANNELS_8)
#     await lh.discard_tips()

In [ ]:
# # ---- PRECISE DISPENSE (more conservative; if you want maximum CV tightness) ----
# precise_dispense = dict(
#     lld_mode=[STARBackend.LLDMode.GAMMA]*8,
#     gamma_lld_sensitivity=[2]*8,
#     # lld_search_height=[4]*8,
#     # minimum_height=[4]*8,
#     # dispensing_mode=[2]*8,               # surface, partial-volume
#     # Slightly deeper contact + a touch slower to bias accuracy over speed:
#     immersion_depth=[0]*8,             # 0.7 mm below surface
#     immersion_depth_direction=[0]*8,
#     # surface_following_distance=[0]*8,
#     transport_air_volume=[0]*8,
#     # Slightly slower and steadier:
#     flow_rates=[10]*8,            # a bit slower than 'dispensing'
#     # cut_off_speed=[8]*8,                 # keep stop abrupt (match flow)
#     # stop_back_volume=[0]*8,
#     swap_speed=[60]*8,                   # slower withdrawal
#     settling_time=[1]*8,               # a hair more dwell for micro-drops
#     jet=[False]*8,
#     blow_out=[False]*8
# )

# # ---- ASPIRATION (Orange G source; single 30 µL draw per channel) ----
# aspirate_OG = dict(
#     lld_mode=[STARBackend.LLDMode.GAMMA]*8,
#     gamma_lld_sensitivity=[2]*8,
#     immersion_depth_direction=[0]*8,
#     flow_rates=[12]*8,           # a bit slower than your 20 baseline
#     settling_time=[1]*8,               # brief pause before lifting out
#     mix_volume=[10]*8,                   # small, below tip max; avoid foaming
#     mix_cycles=[2]*8,
#     mix_speed=[10]*8,
#     transport_air_volume=[0]*8         # no transport air in the tip stack
# )

# async def add_orangeG():
#     await lh.pick_up_tips(tips_10ul["A5:H5"], use_channels=CHANNELS_8)
    
#     for c in range(1, 12, 1): # columns 1-10
#         # Aspirate 10 µL across all 8 channels
#         await lh.aspirate(src_plate["A1"]*8, vols=[10]*8, use_channels=CHANNELS_8, **aspirate_OG)
#         # dispense to src plate to prime pump
#         await lh.dispense(src_plate["A1"]*8, vols=[4]*8, use_channels=CHANNELS_8, **precise_dispense)
#         # dispense to 1st column
#         await lh.dispense(col_wells(dst_plate, c), vols=[4]*8, use_channels=CHANNELS_8, **precise_dispense)
#         # remove residual
#         await lh.dispense(src_plate["A1"]*8, vols=[2]*8, use_channels=CHANNELS_8, lld_mode=[STARBackend.LLDMode.GAMMA]*8, blow_out=[True]*8, empty=[True]*8)
#     # 11th column    
#     # await lh.aspirate(src_plate["A1"]*8, vols=[6]*8, use_channels=CHANNELS_8, **aspirate_OG)
#     # await lh.dispense(col_wells(dst_plate, 11), vols=[4]*8, use_channels=CHANNELS_8, **precise_dispense)
#     # await lh.dispense(src_plate["A1"]*8, vols=[2]*8, use_channels=CHANNELS_8, lld_mode=[STARBackend.LLDMode.GAMMA]*8, blow_out=[True]*8, empty=[True]*8)
#     # return tips
#     # await lh.drop_tips(tips_10ul["A2:H2"], use_channels=CHANNELS_8)
#     await lh.discard_tips()




In [ ]:
# await fill_plate_with_water()
# await add_orangeG()

In [ ]:
# await lh.dispense(src, vols=[50]*8, use_channels=[0], blow_out=[1])
# await lh.blow_out(location=waste["A1"], use_channels=channels)
# await lh.dispense(src_wells, vols=[100]*8, use_channels=USE_CHANS, **precise, liquid_height=[30]*8)   # no blow-out
# # tip_rack = lh.deck.get_resource("htf_tips")
# await lh.drop_tips(htf_tips["A1:H1"], use_channels=list(range(8)))
# await lh.dispense(water_plate["A1"]*8, vols=[10]*8, use_channels=CHANNELS_8, lld_mode=[STARBackend.LLDMode.GAMMA]*8)
# await lh.dispense(src_plate["A1"]*8, vols=[10]*8, use_channels=CHANNELS_8, lld_mode=[STARBackend.LLDMode.GAMMA]*8, blow_out=[True]*8, empty=[True]*8)
# await lh.dispense(src_plate["A1"]*8, vols=[10]*8, use_channels=CHANNELS_8, lld_mode=[STARBackend.LLDMode.GAMMA]*8)
# await lh.dispense(src_plate["A1"]*8, vols=[1]*8, use_channels=CHANNELS_8, liquid_height=[2]*8)
# # await lh.drop_tips(tiprack["A1:C1"])
# await lh.drop_tips(tips_10ul["A1:H1"], use_channels=CHANNELS_8)  # 10 µL filtered tips
# await lh.discard_tips()
# # await lh.prepare_for_manual_channel_operation()
# await lh.stop()